In [1]:
import numpy as np
from matplotlib import pyplot as plt
import pandas as pd
from sklearn.metrics import confusion_matrix
import pynapple as nap
from spatial_manifolds.toroidal import *
from spatial_manifolds.behaviour_plots import *

from spatial_manifolds.mlencoding import *
from spatial_manifolds.circular_decoder import circular_decoder, cross_validate_decoder, cross_validate_decoder_time, circular_nanmean
from spatial_manifolds.data.curation import curate_clusters
from scipy.stats import zscore
from spatial_manifolds.util import gaussian_filter_nan
from spatial_manifolds.predictive_grid import compute_travel_projected, wrap_list
from spatial_manifolds.behaviour_plots import *
from spatial_manifolds.detect_grids import *
from spatial_manifolds.brainrender_helper import *
import seaborn as sns
from scipy.stats import ttest_rel

import warnings
warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [2]:
path = '/Users/harryclark/Downloads/COHORT12/xgboost'
data = pd.DataFrame()
for i, file in enumerate(os.listdir(path)):
    print(f'Reading {file}, {i+1}/{len(os.listdir(path))}')
    file_path = os.path.join(path, file)
    if file.endswith('.pkl'):
        df = pd.read_pickle(file_path)
    elif file.endswith('.csv'):
        df = pd.read_csv(file_path)
    data = pd.concat([data, df], ignore_index=True)

Reading M29_D23_C163.csv, 1/4682
Reading M21_D16_C323.csv, 2/4682
Reading M25_D25_C172.csv, 3/4682
Reading M27_D23_C291.csv, 4/4682
Reading M26_D15_C199.csv, 5/4682
Reading M21_D16_C445.csv, 6/4682
Reading M28_D23_C206.csv, 7/4682
Reading M21_D26_C101.csv, 8/4682
Reading M21_D21_C428.csv, 9/4682
Reading M21_D16_C451.csv, 10/4682
Reading M26_D18_C262.csv, 11/4682
Reading M21_D16_C337.csv, 12/4682
Reading M25_D24_C172.csv, 13/4682
Reading M21_D25_C191.csv, 14/4682
Reading M20_D23_C94.pkl, 15/4682
Reading M29_D20_C29.csv, 16/4682
Reading M25_D19_C94.csv, 17/4682
Reading M21_D21_C414.csv, 18/4682
Reading M29_D25_C476.csv, 19/4682
Reading M20_D15_C356.pkl, 20/4682
Reading M20_D25_C160.pkl, 21/4682
Reading M21_D25_C185.csv, 22/4682
Reading M26_D18_C35.csv, 23/4682
Reading M28_D19_C153.csv, 24/4682
Reading M21_D16_C486.csv, 25/4682
Reading M20_D26_C133.pkl, 26/4682
Reading M21_D16_C492.csv, 27/4682
Reading M20_D26_C127.pkl, 28/4682
Reading M29_D19_C222.csv, 29/4682
Reading M26_D15_C16.csv, 30

In [3]:
# Load session
gcs_ = pd.DataFrame()
ngs_ = pd.DataFrame()
ns_ = pd.DataFrame()
sc_ = pd.DataFrame()
ngs_ns_ = pd.DataFrame()
all_ = pd.DataFrame()

mouse_days = {20: [14,15,16,17,18,19,20,21,22,23,24,25,26],
              21: [15,16,17,18,19,20,21,22,23,24,25,26],
              25: [16,17,18,19,20,21,22,23,24,25],
              26: [11,12,13,14,15,16,17,18,19],
              27: [16,17,18,19,20,21,22,23,24,26],
              28: [16,17,18,19,20,21,22,23,25],
              29: [16,17,18,19,20,21,22,23,25],
            }

for mouse, days in mouse_days.items():
    for day in days:

        # remove M22 
        if mouse == 22:
            continue

        [gcs, ngs, ns, sc, ngs_ns, all] = cell_classification_of1(mouse, day)
        _,_,_,_,_,clusters_VR = compute_vr_tcs(mouse, day)

        gcs = load_cluster_locations(clusters_VR, cells=gcs)
        ngs = load_cluster_locations(clusters_VR, cells=ngs)
        ns = load_cluster_locations(clusters_VR, cells=ns)
        sc = load_cluster_locations(clusters_VR, cells=sc)
        ngs_ns = load_cluster_locations(clusters_VR, cells=ngs_ns)
        all = load_cluster_locations(clusters_VR, cells=all)
        
        gcs_ = pd.concat([gcs_, gcs], ignore_index=True)
        ngs_ = pd.concat([ngs_, ngs], ignore_index=True)
        ns_ = pd.concat([ns_, ns], ignore_index=True)
        sc_ = pd.concat([sc_, sc], ignore_index=True)
        ngs_ns_ = pd.concat([ngs_ns_, ngs_ns], ignore_index=True)
        all_ = pd.concat([all_, all], ignore_index=True)

    print(f'there are this many cells at the moment, {len(all_)}')

20 14
optimal travel lag is 0.2502502502502537 cm based on kde of travel at max spatial information
optimal travel lag is 0.2502502502502537 cm
there are 169 non_grid and non_spatial_cells
there are 0 grid_cells
there are 93 non grid spatial cells
there are 76 non spatial cells
there are 35 speed cells
there are 204 cells
for the non-grid spatial cells the unique locations are ['ENTm1' 'ENTm2' 'ENTm3' 'ENTm5']
20 15
optimal travel lag is 3.753753753753756 cm based on kde of travel at max spatial information
optimal travel lag is 3.753753753753756 cm
there are 63 non_grid and non_spatial_cells
there are 1 grid_cells
there are 52 non grid spatial cells
there are 11 non spatial cells
there are 11 speed cells
there are 75 cells
for the non-grid spatial cells the unique locations are ['ENTm2' 'ENTm3' 'ENTm5']
for the grid cells the unique locations are ['ENTm3']
20 16
optimal travel lag is 10.860860860860868 cm based on kde of travel at max spatial information
optimal travel lag is 10.86086

In [7]:
data[(data['trial_type'] == 'session') &
     (data['ordering'] == 'random') & 
     (data['n_neurons'] == 1)]

,Unnamed: 0,mouse,day,cluster_id,trained_on,tested_on,ordering,n_neurons,pR2_cv,trial_type,trial_context,trial_performance,n_filters,history_length,pR2,order_by
373,373.0,29,23,163,GC,NGS,random,1,NaN,session,session,session,5.0,1000.0,0.013303,NaN
1325,1325.0,29,23,163,NGS,NGS,random,1,NaN,session,session,session,5.0,1000.0,0.049964,NaN
2056,373.0,21,16,323,GC,other,random,1,NaN,session,session,session,5.0,1000.0,0.004353,NaN
3008,1325.0,21,16,323,NGS,other,random,1,NaN,session,session,session,5.0,1000.0,0.014598,NaN
3671,305.0,25,25,172,GC,other,random,1,NaN,session,session,session,5.0,1000.0,0.016029,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7594103,1070.0,29,25,447,NGS,NGS,random,1,NaN,session,session,session,5.0,1000.0,0.004855,NaN
7594664,203.0,26,15,180,GC,other,random,1,NaN,session,session,session,5.0,1000.0,0.002459,NaN
7595463,1002.0,26,15,180,NGS,other,random,1,NaN,session,session,session,5.0,1000.0,0.000464,NaN
7597928,203.0,27,23,288,GC,NGS,random,1,NaN,session,session,session,5.0,1000.0,0.048061,NaN
